# Generation: Generating a Response

In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [3]:
vectorstore = Chroma(persist_directory = "./intro-to-ds-lectures", 
                     embedding_function = OpenAIEmbeddings(model='text-embedding-ada-002'))

In [4]:
len(vectorstore.get()['documents'])

81

In [5]:
retriever = vectorstore.as_retriever(search_type = 'mmr', 
                                     search_kwargs = {'k':3, 
                                                      'lambda_mult':0.7})

In [6]:
TEMPLATE = '''
Answer the following question:
{question}

To answer the question, use only the following context:
{context}

At the end of the response, specify the name of the lecture this context is taken from in the format:
Resources: *Lecture Title*
where *Lecture Title* should be substituted with the title of all resource lectures.
'''

prompt_template = PromptTemplate.from_template(TEMPLATE)

In [7]:
chat = ChatOpenAI(model= 'gpt-4', 
                  seed=365,
                  max_tokens = 250)

In [8]:
question = "What software do data scientists use?"

In [15]:
chain = ({'context': retriever,
         'question': RunnablePassthrough()} 
         | prompt_template 
         | chat
         | StrOutputParser())

In [16]:
chain.invoke(question)

'Data scientists use a variety of programming languages and software in their work. Notably, R and Python are two of the most popular tools used in data science. These languages can manipulate data and integrate it into various data science software platforms. They are adaptable and can tackle a wide range of business and data-related problems from start to finish. In addition, data scientists use software frameworks like Hadoop, which is designed to handle the complexity and computational intensity of big data by distributing tasks across multiple computers. Finally, software such as Power BI, SaS, Qlik, and most notably, Tableau, are used for business intelligence visualizations.\n\nResources: Programming Languages & Software Employed in Data Science - All the Tools You Need'

In [17]:
print('Data scientists use a variety of programming languages and software in their work. Notably, R and Python are two of the most popular tools used due to their ability to manipulate data, perform mathematical and statistical computations, and adapt to a wide variety of business and data-related problems. Additionally, there are also several software frameworks and applications designed specifically for data science. For example, Hadoop is designed to handle the complexity of big data, while Power BI, SaS, Qlik, and Tableau are examples of software designed for business intelligence visualizations.\n\nResources: Programming Languages & Software Employed in Data Science - All the Tools You Need.')

Data scientists use a variety of programming languages and software in their work. Notably, R and Python are two of the most popular tools used due to their ability to manipulate data, perform mathematical and statistical computations, and adapt to a wide variety of business and data-related problems. Additionally, there are also several software frameworks and applications designed specifically for data science. For example, Hadoop is designed to handle the complexity of big data, while Power BI, SaS, Qlik, and Tableau are examples of software designed for business intelligence visualizations.

Resources: Programming Languages & Software Employed in Data Science - All the Tools You Need.
